In [ ]:
# =============================================================================
# PARAMETER REFERENCE — RTX 3060 laptop (6 GB VRAM) + 32 GB RAM
# Model: deepseek-r1:14b (Q4_K_M, ~9 GB). KV cache q8_0 ≈ 96 KB/token.
# RAM ceiling for KV cache before disk swap: ~96k ctx. Stay at or below it.
# =============================================================================
#
# THE FOUR KNOBS
#
#   num_ctx      Context window in tokens. Holds prompt + docs + thinking
#                + final answer. Too small → truncation. Too big → slower
#                prefill and more RAM.
#
#   num_predict  Hard cap on generated tokens (thinking + final answer
#                combined for R1). Cap, not target — model stops when done.
#                Too low → cut-off mid-answer. Too high → no penalty except
#                the theoretical max.
#
#   num_batch    Tokens processed in parallel during prompt ingestion.
#                Does NOT affect generation speed. On 6 GB VRAM, 512 is the
#                ceiling worth using. 256 if you hit OOM. Never raise above
#                512 on this hardware — steals VRAM from model weights.
#
#   temperature  0.6 default. 0.5 for math/algorithmic reasoning (more
#                deterministic). 0.7-0.8 for brainstorming. Leave alone
#                otherwise.
#
# =============================================================================
# QUICK LOOKUP BY TASK (Optimized for RTX 3060 Laptop)
# =============================================================================
#
# TASK                        num_ctx  num_predict  num_batch  temperature
# --------------------------  -------  -----------  ---------  -----------
# Quick chat / fact check        8192         512        512          0.6
# Simple code (< 200 lines)     16384        2048        512          0.6
# Moderate module (~500 lines)  32768        4096        512          0.6
# Complex code / large file     49152        6144        512          0.6
# Single scientific paper       32768        5120        512          0.6
# 2-3 files cross-reference     49152        6144        512          0.6
# 3-4 files, deep analysis      65536        8192        512          0.6
# 5+ files, full review         98304       12288        512          0.6
# Math / derivation / proof     32768        8192        512          0.5
# "Let it cook" deep dive       65536       12288        512          0.5
#
# =============================================================================
# RULES OF THUMB
# =============================================================================
#
# 1. If the answer gets CUT OFF mid-sentence → raise num_predict.
#    For R1, thinking eats budget too. Rough split: ~60% thinking, ~40%
#    answer. So a 4096 cap ≈ 1600 tokens of actual answer.
#
# 2. If the model says "I don't see the full file" or ignores part of a
#    document → raise num_ctx.
#
# 3. If things get SLOWER unexpectedly after a change → you probably pushed
#    num_ctx high enough to spill KV cache to disk. Check `free -h` while
#    running. If swap is being used, drop num_ctx.
#
# 4. Never touch num_batch on this hardware. 512 is the sweet spot.
#
# 5. Switching num_ctx mid-session re-evaluates the ENTIRE history at the
#    new size. Expect a pause on the first call after a switch. If you're
#    doing heavy work, start at the high value and stay there.
#
# 6. Rule of thumb for num_ctx given N tokens of input:
#       num_ctx ≈ input_tokens + num_predict + 512 (safety)
#    Characters ÷ 4 ≈ tokens for English. So a 30,000-char doc ≈ 7,500 tok.
#
# =============================================================================
# EXAMPLE CALLS
# =============================================================================
#
# Quick question:
#   ai.chat(prompt="What is OBIA?",
#           num_ctx=8192, num_predict=512)
#
# Review one large notebook:
#   ai.chat(prompt="Check this for bugs and edge cases.",
#           file_names=["code.ipynb"],
#           num_ctx=49152, num_predict=6144)
#
# Cross-reference three scripts:
#   ai.chat(prompt="Do these agree on the CRS convention?",
#           file_names=["load.py", "reproject.py", "export.py"],
#           num_ctx=65536, num_predict=8192)
#
# Derive something (lower temp):
#   ai.chat(prompt="Derive the gradient of this loss w.r.t. weights.",
#           num_ctx=32768, num_predict=8192, temperature=0.5)
#
# =============================================================================

In [5]:
from main import ScientificAssistant

# Test

In [11]:
assistant = ScientificAssistant(model_name="deepseek-r1:14b")
assistant.clear_memory()
assistant.chat(
    prompt="""
    Just say hi. I am checking that you are there.
    """,
    num_ctx=128,
    num_batch=128,
    num_predict=128
)

🧠 Assistant memory cleared.
🤖 ctx=128 predict=128 batch=128 temp=0.6 | 0 file(s) attached



### 🧠 Thought Process

Alright, the user just said, "Hi. I am checking that you are there." Sounds like they're testing if I'm responsive. I should keep it simple and friendly.

I'll respond with a greeting and offer help. Maybe add an emoji to keep it light. Let's see, "Hello! I'm here and ready to help. How can I assist you today?" That should do it.


---
### 💡 Final Response


Hello! I'm here and ready to help. How can I assist you today?



✅ Turn Complete.
💾 Saved 2 messages and 0 documents to outputs/chat_history/chat_history.json
